# SLAE-CS on a Colab GPU

Installs SLAE-CS, checks the installation against the reference output of the paper's environment, and
predicts backbone amide ¹⁵N chemical shifts (SLAE-CS, 10-seed ensemble) and their expected error
(SLAE-ERROR) for a protein structure.

**Runtime → Change runtime type → GPU** (a T4 is enough). The install takes a few minutes.

Code: [WaymentSteeleLab/SLAE-CS](https://github.com/WaymentSteeleLab/SLAE-CS) · weights:
[hws-lab/SLAE-CS](https://huggingface.co/hws-lab/SLAE-CS)

## 1. Code and environment

*While the repositories are private:* add two Colab secrets (key icon in the left panel, *Notebook access* on):
`GH_TOKEN`, a GitHub token that can read the code repository, and `HF_TOKEN`, a Hugging Face token that can
read `hws-lab/SLAE-CS`.

**Without any token:** put the two zips you were given in `/content` (file panel on the left), or in your
Google Drive and run the optional cell below. `SLAE-CS.zip` replaces the clone and `SLAE-CS-weights.zip`
replaces the download of the weights.

In [ ]:
# Optional: copy the two zips from your Google Drive (faster than uploading 640 MB through the browser).
# from google.colab import drive
# drive.mount("/content/drive")
# !cp /content/drive/MyDrive/SLAE-CS.zip /content/drive/MyDrive/SLAE-CS-weights.zip /content/

In [ ]:
!nvidia-smi -L

In [ ]:
import base64, os, subprocess
try:
    from google.colab import userdata
    for k in ("GH_TOKEN", "HF_TOKEN"):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except ImportError:
    pass

REPO = "https://github.com/WaymentSteeleLab/SLAE-CS.git"
ZIP = "/content/SLAE-CS.zip"     # alternative to cloning: the repository as a zip, uploaded to /content
if not os.path.isdir("SLAE-CS") and os.path.exists(ZIP):
    import zipfile
    zipfile.ZipFile(ZIP).extractall(".")
    print("using the code in", ZIP)
if not os.path.isdir("SLAE-CS"):
    cmd = ["git", "clone", "-q", REPO, "SLAE-CS"]
    token = os.environ.get("GH_TOKEN")
    if token:   # sent as a header, so the token is not part of the URL, the error messages or .git/config
        auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        cmd[1:1] = ["-c", f"http.extraHeader=Authorization: Basic {auth}"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        msg = r.stderr.replace(token, "***") if token else r.stderr
        raise SystemExit("git clone failed:\n" + msg + "\nWhile the repository is private, GH_TOKEN must be a token "
                         "that can read WaymentSteeleLab/SLAE-CS (resource owner WaymentSteeleLab, Contents: read-only; "
                         "an organisation owner may first have to approve it).")
%cd SLAE-CS
!git log -1 --format='%h %s' 2>/dev/null || echo "code from SLAE-CS.zip"

In [ ]:
# A separate Python 3.11 environment with the versions of the paper's environment
# (containers/slae/pip_list.txt), CUDA 12.4 builds. Colab's own Python is newer than these packages support.
# python-dotenv: needed by atomworks 1.0.3 but not declared by it.
!pip install -q uv
!uv venv -q --seed --python 3.11 /content/slae-env
PY = "/content/slae-env/bin/python"
!{PY} -m pip install -q torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
!{PY} -m pip install -q torch-scatter==2.1.2 torch-cluster==1.6.3 torch-sparse==0.6.18 -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
!{PY} -m pip install -q torch==2.6.0 torch-geometric==2.7.0 e3nn==0.5.8 einops==0.8.1 einx==0.3.0 esm==3.2.1 \
    lightning==2.5.6 hydra-core==1.3.2 omegaconf==2.3.0 loguru==0.7.3 opt-einsum-fx==0.1.4 \
    torch-runstats==0.2.0 biotite==1.5.0 atomworks==1.0.3 python-dotenv==1.2.1 safetensors==0.6.2 \
    huggingface-hub==0.36.0 numpy==1.26.4
!{PY} -c "import torch, torch_geometric; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| PyG', torch_geometric.__version__)"

## 2. Weights and installation test

In [ ]:
WEIGHTS_ZIP = "/content/SLAE-CS-weights.zip"      # the weights as a zip: no Hugging Face token needed
if not os.path.isdir("model") and os.path.exists(WEIGHTS_ZIP):
    import zipfile
    zipfile.ZipFile(WEIGHTS_ZIP).extractall(".")
    print("weights from", WEIGHTS_ZIP)
if not os.path.isdir("model"):
    subprocess.run([PY, "scripts/download_from_hf.py", "--no-data"], check=True)   # weights only, into model/
import glob
n = len(glob.glob("model/SLAE-CS-PANAV/seed*/model.safetensors"))
assert n == 10 and os.path.exists("model/SLAE-ERROR/model.safetensors"), f"incomplete weights: {n} SLAE-CS models"
print("10 SLAE-CS models + SLAE-ERROR in", os.path.abspath("model"))

In [ ]:
# predict.py on the three structures in examples/, compared with the reference output (NVIDIA A100)
!{PY} tests/smoke_test.py --device cuda

## 3. Your own structure

One protein per PDB file. Upload a file, or keep `PDB` pointing at one of the examples. Only the first model
of a multi-model file is used.

In [ ]:
PDB = "examples/structures/15419_1.pdb"
try:
    from google.colab import files
    up = files.upload()                      # cancel to keep the example
    if up:
        PDB = next(iter(up))
except Exception:
    pass
print("structure:", PDB)
!{PY} predict.py --weights model/SLAE-CS-PANAV --error-model model/SLAE-ERROR --pdb "$PDB" --out prediction --device cuda

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("prediction.csv")
print(f"{df.protein.nunique()} protein(s), {len(df)} residues (prolines have no amide N and are left out)")
display(df[["protein", "chain", "resnum", "resname", "N_mean", "N_std", "N_abs_error"]].head(10))

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
a1.errorbar(df.resnum, df.N_mean, yerr=df.N_std, fmt="o", ms=3, lw=0.8, capsize=0)
a1.invert_yaxis()
a1.set_ylabel("predicted 15N (ppm)")
a1.set_title("SLAE-CS: mean of 10 seeds (bars: spread between seeds)")
a2.bar(df.resnum, df.N_abs_error, width=0.8)
a2.set_ylabel("expected |error| (ppm)")
a2.set_xlabel("residue number")
a2.set_title("SLAE-ERROR")
plt.tight_layout()
plt.show()

In [ ]:
try:
    from google.colab import files
    files.download("prediction.csv")
    files.download("prediction.json")
except Exception:
    print("prediction.csv and prediction.json are in", os.getcwd())